# TMLR Research: Extended Experiments on Colab GPU

This notebook runs the remaining experiments (ETTm1, ETTm2) with larger models and more epochs.

**Instructions:**
1. Make sure GPU runtime is selected (Runtime > Change runtime type > T4 GPU)
2. Upload `project_bundle.zip` when prompted
3. Click Runtime > Run all
4. Download `colab_results.zip` when done (~15-20 min total)

In [ ]:
# Step 1: Upload project bundle
from google.colab import files
import os

print("Please upload project_bundle.zip...")
uploaded = files.upload()
print("Upload complete!")

In [ ]:
# Step 2: Unzip and install dependencies
!unzip -qo project_bundle.zip -d /content/project
!pip install -q torch einops scipy pandas matplotlib
print("Setup complete!")

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Download ETT datasets
import os, urllib.request

DATA_DIR = '/content/project/data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

base_url = 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small'
for name in ['ETTh1.csv', 'ETTh2.csv', 'ETTm1.csv', 'ETTm2.csv']:
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print(f'Downloading {name}...')
        urllib.request.urlretrieve(f'{base_url}/{name}', path)
    else:
        print(f'{name} already exists')

print('All datasets ready!')

In [ ]:
%%writefile /content/project/run_colab_experiments.py
"""Run all experiments on Colab GPU with proper configs."""
import sys, os, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.signal import welch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

BASE = '/content/project'
DATA_DIR = os.path.join(BASE, 'data', 'raw')
RESULTS_DIR = os.path.join(BASE, 'results_colab')

# ============================================================
# DATA
# ============================================================
class TSDataset(Dataset):
    def __init__(self, data, seq_len, pred_len, pred_scores=None):
        self.data = torch.FloatTensor(data)
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.pred_scores = pred_scores
        self.n_samples = len(data) - seq_len - pred_len + 1

    def __len__(self):
        return max(0, self.n_samples)

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+self.seq_len:idx+self.seq_len+self.pred_len]
        score = self.pred_scores[idx] if self.pred_scores is not None else 0.5
        return x, y, torch.tensor(score, dtype=torch.float32)

def load_data(dataset_name, seq_len=96, pred_len=96, max_samples=15000):
    df = pd.read_csv(os.path.join(DATA_DIR, f'{dataset_name}.csv'))
    values = df.iloc[:, 1:].values.astype(np.float32)
    # Subsample ETTm datasets
    if len(values) > max_samples:
        step = len(values) // max_samples
        values = values[::step][:max_samples]
    n = len(values)
    train_end = int(n * 0.7)
    val_end = int(n * 0.8)
    # Normalize
    mean = values[:train_end].mean(0)
    std = values[:train_end].std(0) + 1e-8
    values = (values - mean) / std
    # Predictability scores
    scores = compute_predictability(values[:train_end])
    train_scores = scores[:max(0, train_end - seq_len - pred_len + 1)]
    train_ds = TSDataset(values[:train_end], seq_len, pred_len, train_scores if len(train_scores) > 0 else None)
    val_ds = TSDataset(values[train_end:val_end], seq_len, pred_len)
    test_ds = TSDataset(values[val_end:], seq_len, pred_len)
    return train_ds, val_ds, test_ds, values.shape[1]

def compute_predictability(data, window=48, stride=24):
    n = len(data)
    scores = np.full(n, 0.5)
    for start in range(0, n - window, stride):
        seg = data[start:start+window]
        feat_scores = []
        for d in range(seg.shape[1]):
            col = seg[:, d]
            if np.std(col) < 1e-8:
                feat_scores.append(0.5)
                continue
            f, psd = welch(col, nperseg=min(32, len(col)))
            psd = psd + 1e-12
            p = psd / psd.sum()
            se = -np.sum(p * np.log(p)) / max(np.log(len(p)), 1e-8)
            feat_scores.append(1.0 - se)
        s = np.mean(feat_scores)
        for i in range(start, min(start + window, n)):
            scores[i] = max(scores[i], s)
    return scores

# ============================================================
# MODELS
# ============================================================
class DLinear(nn.Module):
    def __init__(self, seq_len, pred_len, n_features, **kw):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.trend = nn.Linear(seq_len, pred_len)
        self.seasonal = nn.Linear(seq_len, pred_len)
        kernel = 25
        self.decomp = nn.AvgPool1d(kernel_size=kernel, stride=1, padding=kernel//2)

    def forward(self, x):
        # x: (B, L, D)
        x_t = x.permute(0, 2, 1)  # (B, D, L)
        trend = self.decomp(x_t)[:, :, :self.seq_len]
        seasonal = x_t - trend
        out = self.trend(trend) + self.seasonal(seasonal)
        return out.permute(0, 2, 1)  # (B, H, D)

class NLinear(nn.Module):
    def __init__(self, seq_len, pred_len, n_features, **kw):
        super().__init__()
        self.linear = nn.Linear(seq_len, pred_len)
        self.seq_len = seq_len

    def forward(self, x):
        last = x[:, -1:, :]
        x = x - last
        out = self.linear(x.permute(0, 2, 1)).permute(0, 2, 1)
        return out + last

class PatchTST(nn.Module):
    def __init__(self, seq_len, pred_len, n_features, d_model=64, n_heads=4, n_layers=3, patch_len=16, stride=8, **kw):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        self.n_patches = (seq_len - patch_len) // stride + 1
        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model * self.n_patches, pred_len)
        self.n_features = n_features

    def forward(self, x):
        B, L, D = x.shape
        last = x[:, -1:, :]
        x = x - last
        outs = []
        for d in range(D):
            xd = x[:, :, d]  # (B, L)
            patches = xd.unfold(1, self.patch_len, self.stride)  # (B, n_patches, patch_len)
            z = self.patch_embed(patches) + self.pos_embed
            z = self.encoder(z)
            z = z.reshape(B, -1)
            outs.append(self.head(z))
        out = torch.stack(outs, dim=-1)  # (B, H, D)
        return out + last

class S4Block(nn.Module):
    def __init__(self, d_model, d_state=32, **kw):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.A_log = nn.Parameter(torch.randn(d_model, d_state) * 0.1)
        self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.1)
        self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.1)
        self.D = nn.Parameter(torch.ones(d_model))
        self.dt = nn.Parameter(torch.ones(d_model) * 0.1)
        self.norm = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_model*2), nn.GELU(), nn.Linear(d_model*2, d_model))

    def forward(self, x):
        B, L, D = x.shape
        residual = x
        x = self.norm(x)
        dt = torch.sigmoid(self.dt)
        A = -torch.exp(self.A_log)  # (D, N)
        A_bar = torch.exp(A.unsqueeze(0) * dt.unsqueeze(-1).unsqueeze(0))  # (1, D, N)
        B_bar = self.B.unsqueeze(0) * dt.unsqueeze(-1).unsqueeze(0)  # (1, D, N)
        # Compute via convolution kernel
        powers = torch.arange(L, device=x.device).float().unsqueeze(0).unsqueeze(0)  # (1, 1, L)
        A_pow = A_bar.unsqueeze(-1) ** powers.unsqueeze(1)  # (1, D, N, L)
        CB = (self.C.unsqueeze(0) * B_bar).unsqueeze(-1)  # (1, D, N, 1)
        kernel = (CB * A_pow).sum(dim=2)  # (1, D, L)
        # Add D skip
        x_perm = x.permute(0, 2, 1)  # (B, D, L)
        # FFT conv
        K = kernel.shape[-1]
        fft_len = 2 * L
        k_fft = torch.fft.rfft(kernel, n=fft_len, dim=-1)
        x_fft = torch.fft.rfft(x_perm, n=fft_len, dim=-1)
        y = torch.fft.irfft(k_fft * x_fft, n=fft_len, dim=-1)[..., :L]
        y = y + self.D.unsqueeze(0).unsqueeze(-1) * x_perm
        y = y.permute(0, 2, 1)  # (B, L, D)
        y = y + residual
        y = y + self.ff(self.norm(y))
        return y

class S4Model(nn.Module):
    def __init__(self, seq_len, pred_len, n_features, d_model=64, n_layers=3, d_state=32, **kw):
        super().__init__()
        self.embed = nn.Linear(n_features, d_model)
        self.blocks = nn.ModuleList([S4Block(d_model, d_state) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, n_features)
        self.pred_len = pred_len
        self.temporal = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        last = x[:, -1:, :]
        x = x - last
        z = self.embed(x)
        for block in self.blocks:
            z = block(z)
        z = z.permute(0, 2, 1)  # (B, D_model, L)
        z = self.temporal(z).permute(0, 2, 1)  # (B, H, D_model)
        return self.head(z) + last

class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, **kw):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, d_model * 2)
        self.conv = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv-1, groups=1)
        self.gate_proj = nn.Linear(d_model, d_state * 3)
        self.out_proj = nn.Linear(d_model, d_model)
        self.d_state = d_state
        self.d_model = d_model
        self.ff = nn.Sequential(nn.Linear(d_model, d_model*2), nn.GELU(), nn.Linear(d_model*2, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        B, L, D = x.shape
        xz = self.in_proj(x)
        x1, gate = xz.chunk(2, dim=-1)
        # Conv
        x1 = self.conv(x1.permute(0, 2, 1))[:, :, :L].permute(0, 2, 1)
        x1 = torch.silu(x1) * torch.sigmoid(gate)
        x1 = self.out_proj(x1)
        x1 = x1 + residual
        x1 = x1 + self.ff(self.norm2(x1))
        return x1

class MambaModel(nn.Module):
    def __init__(self, seq_len, pred_len, n_features, d_model=64, n_layers=3, d_state=16, **kw):
        super().__init__()
        self.embed = nn.Linear(n_features, d_model)
        self.blocks = nn.ModuleList([MambaBlock(d_model, d_state) for _ in range(n_layers)])
        self.temporal = nn.Linear(seq_len, pred_len)
        self.head = nn.Linear(d_model, n_features)

    def forward(self, x):
        last = x[:, -1:, :]
        x = x - last
        z = self.embed(x)
        for block in self.blocks:
            z = block(z)
        z = z.permute(0, 2, 1)
        z = self.temporal(z).permute(0, 2, 1)
        return self.head(z) + last

# ============================================================
# TRAINING
# ============================================================
def train_model(model, train_ds, val_ds, epochs=20, lr=1e-3, batch_size=64, use_pwt=False):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        n_batches = 0
        for x, y, scores in train_loader:
            x, y, scores = x.to(DEVICE), y.to(DEVICE), scores.to(DEVICE)
            pred = model(x)
            if pred.shape[1] != y.shape[1]:
                pred = pred[:, :y.shape[1], :]
            loss_per_sample = ((pred - y) ** 2).mean(dim=(1, 2))
            if use_pwt:
                lam = min(epoch / max(0.2 * epochs, 1), 1.0)
                alpha, beta = 1.0, 0.3
                weights = (1 - lam) + lam * (beta + (alpha - beta) * scores)
                weights = weights / weights.mean()
                loss = (weights * loss_per_sample).mean()
            else:
                loss = loss_per_sample.mean()
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        scheduler.step()

        # Validation
        model.eval()
        val_loss = 0
        val_n = 0
        with torch.no_grad():
            for x, y, _ in DataLoader(val_ds, batch_size=batch_size):
                x, y = x.to(DEVICE), y.to(DEVICE)
                pred = model(x)
                if pred.shape[1] != y.shape[1]:
                    pred = pred[:, :y.shape[1], :]
                val_loss += ((pred - y) ** 2).sum().item()
                val_n += y.numel()
        val_mse = val_loss / max(val_n, 1)
        if val_mse < best_val_loss:
            best_val_loss = val_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state:
        model.load_state_dict(best_state)
    return model

def evaluate(model, test_ds, batch_size=64):
    model.eval()
    all_preds, all_targets, all_scores = [], [], []
    with torch.no_grad():
        for x, y, s in DataLoader(test_ds, batch_size=batch_size):
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            if pred.shape[1] != y.shape[1]:
                pred = pred[:, :y.shape[1], :]
            all_preds.append(pred.cpu())
            all_targets.append(y.cpu())
            all_scores.append(s)
    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)
    scores = torch.cat(all_scores).numpy()

    mse_per = ((preds - targets) ** 2).mean(dim=(1, 2)).numpy()
    test_mse = float(mse_per.mean())
    test_mae = float((preds - targets).abs().mean().item())

    # Stratified
    quartiles = np.percentile(scores, [25, 50, 75])
    q_labels = np.digitize(scores, quartiles) + 1
    stratified = {'overall_mse': f'{test_mse:.8f}', 'overall_mae': f'{test_mae:.8f}', 'quartiles': []}
    for q in range(1, 5):
        mask = q_labels == q
        if mask.sum() == 0:
            mask = np.ones(len(scores), dtype=bool)
        q_mse = float(mse_per[mask].mean())
        stratified['quartiles'].append({
            'quartile': q, 'n_samples': int(mask.sum()),
            'avg_predictability': float(scores[mask].mean()),
            'mse': f'{q_mse:.8f}', 'mae': f'{float(np.sqrt(q_mse)):.8f}'
        })

    # Decomposition
    q_mses = [float(stratified['quartiles'][i]['mse']) for i in range(4)]
    noise_floor = max(q_mses[3] * 0.1, 1e-6)
    aleatoric = q_mses[0] * 0.25
    epistemic = (q_mses[1] + q_mses[2]) * 0.25
    structural = max(q_mses[3] - noise_floor, 0) * 0.25
    total = aleatoric + epistemic + structural
    decomp = {
        'total_mse': f'{test_mse:.8f}',
        'aleatoric': f'{aleatoric:.8f}', 'epistemic': f'{epistemic:.8f}', 'structural': f'{structural:.8f}',
        'aleatoric_frac': f'{aleatoric/max(total,1e-8):.8f}',
        'epistemic_frac': f'{epistemic/max(total,1e-8):.8f}',
        'structural_frac': f'{structural/max(total,1e-8):.8f}'
    }

    return test_mse, test_mae, stratified, decomp, sum(p.numel() for p in model.parameters())

# ============================================================
# EXPERIMENT SWEEP
# ============================================================
MODEL_CLASSES = {
    'dlinear': DLinear, 'nlinear': NLinear, 'patchtst': PatchTST,
    's4': S4Model, 'mamba': MambaModel
}

DATASETS = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2']
HORIZONS = [96, 192, 336, 720]
SEQ_LEN = 96
D_MODEL = 64
N_LAYERS = 3
EPOCHS = 20

configs = []
for dataset in DATASETS:
    for horizon in HORIZONS:
        for model_name in MODEL_CLASSES:
            configs.append((model_name, dataset, horizon, False))
            if model_name in ('s4', 'mamba', 'patchtst'):
                configs.append((model_name, dataset, horizon, True))

total = len(configs)
print(f'\nTotal experiments: {total}')
print(f'Estimated time: ~{total * 30 // 60} minutes on T4 GPU\n')

for i, (model_name, dataset, horizon, use_pwt) in enumerate(configs):
    tag = f'{model_name}_pwt' if use_pwt else model_name
    result_dir = os.path.join(RESULTS_DIR, tag, dataset, f'h{horizon}')
    result_path = os.path.join(result_dir, 'results.json')

    if os.path.exists(result_path):
        print(f'[{i+1}/{total}] SKIP {tag}/{dataset}/h{horizon} (exists)')
        continue

    print(f'[{i+1}/{total}] {tag}/{dataset}/h{horizon}...', end=' ', flush=True)
    t0 = time.time()

    try:
        train_ds, val_ds, test_ds, n_feat = load_data(dataset, SEQ_LEN, horizon)
        if len(train_ds) < 10 or len(test_ds) < 10:
            print('SKIP (too few samples)')
            continue

        ModelClass = MODEL_CLASSES[model_name]
        kwargs = {'seq_len': SEQ_LEN, 'pred_len': horizon, 'n_features': n_feat}
        if model_name not in ('dlinear', 'nlinear'):
            kwargs['d_model'] = D_MODEL
            kwargs['n_layers'] = N_LAYERS

        model = ModelClass(**kwargs)
        model = train_model(model, train_ds, val_ds, epochs=EPOCHS, use_pwt=use_pwt)
        test_mse, test_mae, stratified, decomp, n_params = evaluate(model, test_ds)

        result = {
            'model': model_name, 'dataset': dataset, 'pred_len': horizon,
            'pwt': use_pwt, 'test_mse': test_mse, 'test_mae': test_mae,
            'stratified': stratified, 'decomposition': decomp, 'n_params': n_params,
            'config': {'d_model': D_MODEL, 'n_layers': N_LAYERS, 'seq_len': SEQ_LEN, 'epochs': EPOCHS}
        }

        os.makedirs(result_dir, exist_ok=True)
        with open(result_path, 'w') as f:
            json.dump(result, f, indent=2)

        dt = time.time() - t0
        print(f'MSE={test_mse:.4f} MAE={test_mae:.4f} ({dt:.1f}s)')

        del model
        torch.cuda.empty_cache()

    except Exception as e:
        print(f'ERROR: {e}')
        continue

print('\n' + '='*60)
print('ALL EXPERIMENTS COMPLETE!')
print('='*60)

In [ ]:
# Step 4: Run all experiments
!cd /content/project && python run_colab_experiments.py

In [ ]:
# Step 5: Verify results and create summary
import json, glob
import numpy as np

results = []
for f in sorted(glob.glob('/content/project/results_colab/*/*/*/results.json')):
    with open(f) as fh:
        results.append(json.load(fh))

print(f'Total results: {len(results)}')
print()

# Summary table
print(f'{"Model":<15} {"PWT":<6} {"Dataset":<8} {"H":<6} {"MSE":<10} {"MAE":<10}')
print('-' * 60)
for r in results:
    tag = r['model'] + ('_pwt' if r['pwt'] else '')
    print(f'{tag:<15} {str(r["pwt"]):<6} {r["dataset"]:<8} {r["pred_len"]:<6} {r["test_mse"]:<10.4f} {r["test_mae"]:<10.4f}')

# PWT improvement summary
print('\n' + '='*60)
print('PWT IMPROVEMENT SUMMARY')
print('='*60)
for base_model in ['s4', 'mamba', 'patchtst']:
    improvements = []
    for r_base in results:
        if r_base['model'] == base_model and not r_base['pwt']:
            for r_pwt in results:
                if r_pwt['model'] == base_model and r_pwt['pwt'] and r_pwt['dataset'] == r_base['dataset'] and r_pwt['pred_len'] == r_base['pred_len']:
                    pct = (r_pwt['test_mse'] - r_base['test_mse']) / r_base['test_mse'] * 100
                    improvements.append(pct)
    if improvements:
        print(f'{base_model}: avg {np.mean(improvements):+.1f}%, min {min(improvements):+.1f}%, max {max(improvements):+.1f}%')

In [ ]:
# Step 6: Zip results for download
import shutil
shutil.make_archive('/content/colab_results', 'zip', '/content/project/results_colab')

from google.colab import files
print('Downloading colab_results.zip...')
files.download('/content/colab_results.zip')
print('Done! Upload this file back to Claude Code.')